In [1]:
import sys
sys.path.insert(0, '../../../scripts')
from evaluate_rq1 import *

DATA_PATH = '../../../data/hard_triplets_tfidf_dbpedia_canon.json'
CHECKPOINT_PATH = 'sparqlce/models/dbpedia_model_filter_true_fixed.pt'
TOPKS = [3, 5, 10, 20]
BATCH = 32

raw_data = load_triplets(DATA_PATH)
global_corpus, seen_q = build_global_corpus(raw_data)
eval_items = build_eval_items(raw_data, global_corpus, seen_q)
relevance_scores_per_item = build_binary_relevance_maps(eval_items, global_corpus)

print(f'Anchors: {len(eval_items)}, Corpus: {len(global_corpus)}')

/upb/users/z/zahera/profiles/unix/cs/.conda/envs/sparql-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Anchors: 337, Corpus: 1664


In [2]:
# SPARQLCE
evaluate_sparqlce_checkpoint("../../../"+CHECKPOINT_PATH, eval_items, global_corpus, relevance_scores_per_item, TOPKS, batch_size=BATCH)

# TF-IDF
evaluate_tfidf(eval_items, global_corpus, relevance_scores_per_item, TOPKS)

# E5
evaluate_sentence_transformer('intfloat/e5-base-v2', 'E5-base-v2', eval_items, global_corpus, relevance_scores_per_item, TOPKS, transform=lambda q: f'query: {q}')

# Qwen
evaluate_sentence_transformer('Qwen/Qwen2.5-0.5B-Instruct', 'Qwen2.5-0.5B', eval_items, global_corpus, relevance_scores_per_item, TOPKS)

Loading model from ../../../sparqlce/models/dbpedia_model_filter_true_fixed.pt...

SPARQLCE
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.1810  0.0642  0.0847  0.2369  0.3294    0.1810  0.1770
@5     0.1828  0.1054  0.1177  0.2557  0.4125    0.1828  0.1879
@10    0.1917  0.2106  0.1777  0.2761  0.5638    0.1917  0.2232
@20    0.1901  0.4101  0.2340  0.2900  0.7685    0.1901  0.2936

Baseline: TF-IDF
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.0495  0.0222  0.0285  0.0584  0.1424    0.0495  0.0390
@5     0.0386  0.0244  0.0268  0.0653  0.1751    0.0386  0.0359
@10    0.0231  0.0262  0.0213  0.0653  0.1751    0.0231  0.0301
@20    0.0151  0.0295  0.0173  0.0670  0.2047    0.0151  0.0301


No sentence-transformers model found with name Qwen/Qwen2.5-0.5B. Creating a new one with mean pooling.



E5-base-v2
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.0287  0.0134  0.0162  0.0504  0.0801    0.0287  0.0286
@5     0.0226  0.0163  0.0158  0.0524  0.0890    0.0226  0.0267
@10    0.0208  0.0280  0.0192  0.0562  0.1187    0.0208  0.0300
@20    0.0208  0.0498  0.0246  0.0622  0.2047    0.0208  0.0384

Qwen2.5-0.5B
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.0465  0.0183  0.0231  0.0762  0.1009    0.0465  0.0491
@5     0.0398  0.0240  0.0260  0.0803  0.1187    0.0398  0.0469
@10    0.0356  0.0361  0.0314  0.0852  0.1573    0.0356  0.0475
@20    0.0283  0.0519  0.0325  0.0882  0.1988    0.0283  0.0518
